1. Set Context

In [ ]:
USE ROLE ROLE_IS23837;
USE DATABASE DB_IS23837;
CREATE SCHEMA IF NOT EXISTS HOMEWORKS;
USE SCHEMA HOMEWORKS;
USE WAREHOUSE Compute_WH;

2. Create one shared sequence

In [ ]:
CREATE OR REPLACE SEQUENCE conference_seq START = 1 INCREMENT = 1;

3. Create Tables

In [ ]:
CREATE OR REPLACE TABLE speakers (
    speaker_id  NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    first_name  VARCHAR NOT NULL,
    last_name   VARCHAR NOT NULL,
    email       VARCHAR NOT NULL,
    company     VARCHAR,
    bio         VARCHAR,
    is_keynote  BOOLEAN
);

CREATE OR REPLACE TABLE sponsors (
    sponsor_id              NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    sponsor_name            VARCHAR NOT NULL,
    sponsorship_level       VARCHAR,
    sponsorship_amount      FLOAT,
    contract_signed_date    DATE
);

CREATE OR REPLACE TABLE rooms (
    room_id                 NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    room_name               VARCHAR NOT NULL,
    capacity                NUMBER NOT NULL,
    has_projector           BOOLEAN
);

CREATE OR REPLACE TABLE attendees (
    attendee_id             NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    first_name              VARCHAR NOT NULL,
    last_name               VARCHAR NOT NULL,
    email                   VARCHAR NOT NULL,
    registration_fee_paid   FLOAT,
    checked_in              BOOLEAN
);

Tracks references Sponsors (one-to-many; 1 sponsor, many tracks)
Add FK for one-to-many relationships 

In [ ]:
CREATE OR REPLACE TABLE tracks (
    track_id            NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    track_name          VARCHAR NOT NULL,
    track_description   VARCHAR,
    sponsor_id          NUMBER,
    FOREIGN KEY (sponsor_id) REFERENCES sponsors (sponsor_id)
);

Sessions references Speakers and Tracks (one-to-many)

In [ ]:
CREATE OR REPLACE TABLE sessions (
    session_id      NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    title           VARCHAR NOT NULL,
    abstract        VARCHAR,
    speaker_id      NUMBER,
    track_id        NUMBER,
    is_recorded     BOOLEAN,
    FOREIGN KEY (speaker_id) REFERENCES speakers(speaker_id),
    FOREIGN KEY (track_id) REFERENCES tracks(track_id)
);

Schedule references Sessions and Rooms

In [ ]:
CREATE OR REPLACE TABLE schedule (
    schedule_id         NUMBER DEFAULT conference_seq.NEXTVAL PRIMARY KEY,
    session_id          NUMBER NOT NULL,
    room_id             NUMBER NOT NULL,
    start_time          TIMESTAMP_NTZ NOT NULL,
    end_time            TIMESTAMP_NTZ NOT NULL,
    FOREIGN KEY (session_id) REFERENCES sessions(session_id),
    FOREIGN KEY (room_id) REFERENCES rooms(room_id)
);

Junction table for the many-to-many Attendees <-> Sessions relationship

In [ ]:
CREATE OR REPLACE TABLE registrations (
    attendee_id             NUMBER,
    session_id              NUMBER,
    registration_datetime   TIMESTAMP_NTZ NOT NULL,
    attended                BOOLEAN,
    PRIMARY KEY (attendee_id, session_id), ---- composite key
    FOREIGN KEY (attendee_id) REFERENCES attendees(attendee_id),
    FOREIGN KEY (session_id) REFERENCES sessions(session_id)
);

4. Insert data, Parent tables first, followed by child tables

Primary keys are auto-generated by conference_seq, so exact ID values aren't predictable or hardcoded 
anywhere in this notebook. They'll differ depending on how many times the sequence has been called. 

Foreign keys in child tables are looked up by a unique value (email, name) rather than by hardcoded 
number, so every INSERT below works correctly regardless of the current sequence position.

Speakers — parent table, no foreign keys. speaker_id is assigned automatically via conference_seq.

In [ ]:
INSERT INTO speakers (first_name, last_name, email, company, bio, is_keynote) VALUES
    ('Ana', 'Reyes', 'ana.reyes@example.com', 'DataCorp', 'ML researcher and speaker', TRUE),
    ('Ben', 'Cohen', 'ben.cohen@example.com', 'CloudNine Inc', 'Cloud architecture consultant', FALSE);

Sponsors — parent table, no foreign keys. sponsor_id is assigned automatically via conference_seq.

In [ ]:
INSERT INTO sponsors (sponsor_name, sponsorship_level, sponsorship_amount, contract_signed_date) VALUES
    ('CloudNine Inc', 'Gold', 15000.00, '2026-03-01'),
    ('DataCorp', 'Platinum', 25000.00, '2026-02-15');

Rooms — parent table, no foreign keys. room_id is assigned automatically via conference_seq.

In [ ]:
INSERT INTO rooms (room_name, capacity, has_projector) VALUES
    ('Grand Ballroom A', 200, TRUE),
    ('Conference Room 1', 50, TRUE);

Attendees — parent table, no foreign keys. attendee_id is assigned automatically via conference_seq.

In [ ]:

INSERT INTO attendees (first_name, last_name, email, registration_fee_paid, checked_in) VALUES
    ('Miguel', 'Torres', 'miguel.t@example.com', 249.99, FALSE),
    ('Sara', 'Kim', 'sara.kim@example.com', 249.99, TRUE);

Child Tables

Each of the following references one or more parent tables above via foreign key. 
Rather than hardcoding the parent's ID number (which shifts depending on the sequence's current 
value), each INSERT looks up the correct ID using a unique value already inserted above 
(email, name, or title).

Tracks references Sponsors (one-to-many: one sponsor, many tracks). 

sponsor_id is looked up by sponsor_name rather than hardcoded, so this INSERT is correct 
regardless of the sequence's current value.

In [ ]:
INSERT INTO tracks (track_name, track_description, sponsor_id)
SELECT 'AI & Data', 'Sessions on AI and analytics', sponsor_id FROM sponsors WHERE sponsor_name = 'CloudNine Inc'
UNION ALL
SELECT 'Cloud Infrastructure', 'Cloud platforms and architecture', sponsor_id FROM sponsors WHERE sponsor_name = 'DataCorp'
UNION ALL
SELECT 'Data Engineering', 'Pipelines and data platforms', sponsor_id FROM sponsors WHERE sponsor_name = 'CloudNine Inc'
UNION ALL
SELECT 'Security', 'Cybersecurity and compliance', sponsor_id FROM sponsors WHERE sponsor_name = 'DataCorp';

Sessions references Speakers and Tracks (one-to-many from both).

speaker_id and track_id are looked up by the speaker's email and the track's name respectively, 
rather than hardcoded, so this INSERT is correct regardless of the sequence's current value.

In [ ]:
INSERT INTO sessions (title, abstract, speaker_id, track_id, is_recorded)
SELECT 'Intro to LLMs', 'A beginner-friendly overview of large language models', sp.speaker_id, tr.track_id, TRUE
FROM speakers sp, tracks tr WHERE sp.email = 'ana.reyes@example.com' AND tr.track_name = 'AI & Data'
UNION ALL
SELECT 'Scaling on the Cloud', 'Patterns for elastic cloud infrastructure', sp.speaker_id, tr.track_id, FALSE
FROM speakers sp, tracks tr WHERE sp.email = 'ben.cohen@example.com' AND tr.track_name = 'Cloud Infrastructure'
UNION ALL
SELECT 'Modern Data Pipelines', 'Designing scalable data engineering pipelines', sp.speaker_id, tr.track_id, TRUE
FROM speakers sp, tracks tr WHERE sp.email = 'ana.reyes@example.com' AND tr.track_name = 'Data Engineering'
UNION ALL
SELECT 'Securing Cloud Workloads', 'Practical cloud security techniques', sp.speaker_id, tr.track_id, FALSE
FROM speakers sp, tracks tr WHERE sp.email = 'ben.cohen@example.com' AND tr.track_name = 'Security';

Schedule references Sessions and Rooms.

session_id and room_id are looked up by the session's title and the room's name respectively, 
rather than hardcoded, so this INSERT is correct regardless of the sequence's current value.

In [ ]:
INSERT INTO schedule (session_id, room_id, start_time, end_time)
SELECT s.session_id, r.room_id, '2026-10-12 09:00:00', '2026-10-12 10:00:00'
FROM sessions s, rooms r WHERE s.title = 'Intro to LLMs' AND r.room_name = 'Grand Ballroom A'
UNION ALL
SELECT s.session_id, r.room_id, '2026-10-12 10:15:00', '2026-10-12 11:15:00'
FROM sessions s, rooms r WHERE s.title = 'Scaling on the Cloud' AND r.room_name = 'Conference Room 1'
UNION ALL
SELECT s.session_id, r.room_id, '2026-10-13 09:00:00', '2026-10-13 10:00:00'
FROM sessions s, rooms r WHERE s.title = 'Modern Data Pipelines' AND r.room_name = 'Grand Ballroom A'
UNION ALL
SELECT s.session_id, r.room_id, '2026-10-13 10:15:00', '2026-10-13 11:15:00'
FROM sessions s, rooms r WHERE s.title = 'Securing Cloud Workloads' AND r.room_name = 'Conference Room 1';

Junction table for the many-to-many Attendees <-> Sessions relationship. Composite primary key 
(attendee_id, session_id) — no sequence used here, since this key is made of the two foreign keys.

attendee_id and session_id are looked up by the attendee's email and the session's title 
respectively, rather than hardcoded, so this INSERT is correct regardless of the sequence's 
current value.

In [ ]:
INSERT INTO registrations (attendee_id, session_id, registration_datetime, attended)
SELECT a.attendee_id, s.session_id, '2026-09-01 08:30:00', FALSE
FROM attendees a, sessions s WHERE a.email = 'miguel.t@example.com' AND s.title = 'Intro to LLMs'
UNION ALL
SELECT a.attendee_id, s.session_id, '2026-09-01 08:31:00', TRUE
FROM attendees a, sessions s WHERE a.email = 'sara.kim@example.com' AND s.title = 'Scaling on the Cloud'
UNION ALL
SELECT a.attendee_id, s.session_id, '2026-09-02 09:00:00', FALSE
FROM attendees a, sessions s WHERE a.email = 'miguel.t@example.com' AND s.title = 'Modern Data Pipelines'
UNION ALL
SELECT a.attendee_id, s.session_id, '2026-09-02 09:01:00', TRUE
FROM attendees a, sessions s WHERE a.email = 'sara.kim@example.com' AND s.title = 'Securing Cloud Workloads';

In [ ]:
SELECT * FROM speakers;

In [ ]:
SELECT * FROM sponsors;

In [ ]:
SELECT * FROM rooms;

In [ ]:
SELECT * FROM attendees;

In [ ]:
SELECT * FROM tracks;

In [ ]:
SELECT * FROM sessions;

In [ ]:
SELECT * FROM schedule;

In [ ]:
SELECT * FROM registrations;